# Interpretable Chest X-Ray Pneumonia Detection

**Author:** Your Name  
**Date:** 2026  

---

## Executive Summary

This notebook presents an end-to-end pipeline for classifying chest X-rays as **Normal** or **Pneumonia** using transfer learning (ResNet50), with a strong focus on **explainability** via **Grad-CAM** and **Eigen-CAM** heatmaps.

**Key goals:**
- High F1-score and Precision-Recall performance (not just accuracy)
- Handle severe class imbalance (~25% Normal, ~75% Pneumonia)
- Provide visual explanations clinicians can interpret

> ⚠️ **Disclaimer:** This model is for research and education only. It is **not** approved for clinical diagnosis.

## 1. Setup & Imports

**Colab users:** Run cells **top to bottom**. If `drive.mount` fails in Cursor: **Cmd+Shift+P** → **Colab: Mount Google Drive to Server**, then re-run cell 2. Project on Drive: `My Drive/chest_xray_pneumonia_detection/`

In [ ]:
# Paths & environment (local Mac vs Google Colab)
import os
import sys
from pathlib import Path

def _running_in_colab() -> bool:
    if os.environ.get("COLAB_GPU") or os.environ.get("COLAB_RELEASE_TAG"):
        return True
    if Path("/content").exists() and not Path("/Applications").exists():
        return True  # typical Colab VM layout
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _running_in_colab()

# Change this if your Google Drive folder has a different name
DRIVE_FOLDER = "chest_xray_pneumonia_detection"

if IN_COLAB:
    # drive.mount() often fails in Cursor — use Cmd+Shift+P → Colab: Mount Google Drive to Server
    _mydrive = Path("/content/drive/MyDrive")
    if not (_mydrive.exists() and any(_mydrive.iterdir())):
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("drive.mount() skipped:", e)
            print("Run: Cmd+Shift+P → Colab: Mount Google Drive to Server, then re-run this cell")

    _here = Path.cwd()
    candidates = [
        Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}"),
        Path("/content/drive/MyDrive/Chext_X Pneumonia detection/chest_xray_pneumonia_detection"),
        Path("/content/drive/MyDrive/OTU WINTER 2026/Project 3/Chext_X Pneumonia detection/chest_xray_pneumonia_detection"),
        _here.parent if _here.name == "notebook" else _here,
        Path("/content/chest_xray_pneumonia_detection"),
    ]
    def _has_src(p):
        return (p / "src" / "train.py").exists()

    def _search_mydrive(max_depth=5):
        root = Path("/content/drive/MyDrive")
        skip = {".git", ".venv", "chest_xray", "train", "test", "val", "__pycache__", "NORMAL", "PNEUMONIA"}
        queue = [(root, 0)]
        while queue:
            folder, depth = queue.pop(0)
            if _has_src(folder):
                return folder.resolve()
            if depth >= max_depth:
                continue
            try:
                kids = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".") and c.name not in skip]
            except (OSError, PermissionError):
                continue
            for c in sorted(kids):
                queue.append((c, depth + 1))
        return None

    candidates.insert(1, Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}/{DRIVE_FOLDER}"))
    PROJECT_ROOT = next((p for p in candidates if _has_src(p)), None)
    if PROJECT_ROOT is None:
        PROJECT_ROOT = _search_mydrive()
        if PROJECT_ROOT:
            print("Auto-found project:", PROJECT_ROOT)
    if PROJECT_ROOT is None:
        PROJECT_ROOT = candidates[0]
else:
    # Local: notebook is in notebook/ → project root is parent folder
    PROJECT_ROOT = Path("..").resolve()
    if not (PROJECT_ROOT / "src").exists():
        PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "chest_xray"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (FIGURES_DIR, MODELS_DIR, DATA_ROOT.parent):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

BATCH_SIZE = 16 if IN_COLAB else 32  # reduce to 8 on Colab if CUDA OOM

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:   ", DATA_ROOT)
print("MODELS_DIR:  ", MODELS_DIR)
print("src/ exists:  ", (PROJECT_ROOT / "src").exists())

if IN_COLAB and not (PROJECT_ROOT / "src" / "train.py").exists():
    print("Contents of", PROJECT_ROOT, ":")
    if PROJECT_ROOT.exists():
        for x in sorted(PROJECT_ROOT.iterdir())[:20]:
            print(" ", x.name)
    raise FileNotFoundError(
        f"src/ missing on Drive. Upload the FULL project zip to:\n"
        f"  My Drive/{DRIVE_FOLDER}/\n"
        f"Must include: src/train.py, src/dataset.py, ..."
    )

In [ ]:
# Install dependencies on Colab (safe to re-run)
if IN_COLAB:
    %pip install -q torch torchvision timm grad-cam scikit-learn matplotlib seaborn tqdm Pillow kaggle

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from src.dataset import get_dataset_stats, get_dataloaders, get_transforms, CLASS_NAMES, denormalize
from src.model import build_model, count_parameters, get_target_layer
from src.train import train
from src.evaluate import run_evaluation, plot_training_curves
from src.cam import generate_cam_comparison, plot_cam_comparison, generate_gradcam, generate_eigencam, overlay_heatmap

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if IN_COLAB and not torch.cuda.is_available():
    print('Tip: Runtime → Change runtime type → GPU (or pick Colab GPU kernel)')

## 2. Dataset Exploration

We use the [Kaggle Chest X-Ray Pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset with the **official train/val/test splits**.

**Local:** `python scripts/download_dataset.py`  
**Colab:** Run the next cell once (upload `kaggle.json` when prompted).

In [ ]:
# Download dataset on Colab if missing (skip if DATA_ROOT/train already exists)
if IN_COLAB and not (DATA_ROOT / "train").exists():
    import shutil
    from google.colab import files

    print("Dataset not found. Set up Kaggle API:")
    print("  Kaggle → Account → Create New Token → upload kaggle.json")
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "wb") as f:
            f.write(uploaded["kaggle.json"])
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Downloading from Kaggle (~1.2 GB)...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_ROOT.parent} --unzip
    print("Download complete.")
elif not (DATA_ROOT / "train").exists():
    print(f"Dataset missing at {DATA_ROOT}")
    print("Run: python scripts/download_dataset.py")
else:
    print(f"Dataset OK: {DATA_ROOT}")

stats = get_dataset_stats(DATA_ROOT)
print(json.dumps(stats, indent=2))

In [ ]:
# Visualize class distribution
rows = []
for split, counts in stats.items():
    for cls, n in counts.items():
        rows.append({'Split': split, 'Class': cls, 'Count': n})

df_stats = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_stats, x='Split', y='Count', hue='Class', ax=ax)
ax.set_title('Class Distribution by Split')
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Sample images from each class
from PIL import Image
from src.dataset import ChestXRayDataset

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(CLASS_NAMES):
    ds = ChestXRayDataset(DATA_ROOT, split='train', transform=None)
    samples = [p for p, l in ds.samples if CLASS_NAMES[l] == cls][:4]
    for col, path in enumerate(samples):
        img = Image.open(path).convert('RGB')
        axes[row, col].imshow(img, cmap='gray' if img.mode == 'L' else None)
        axes[row, col].set_title(cls if col == 0 else '')
        axes[row, col].axis('off')
plt.suptitle('Sample Chest X-Rays (Train)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_xrays.png', dpi=150)
plt.show()

## 3. Methodology

### 3.1 Preprocessing
- Resize to 224×224 (ImageNet standard for transfer learning)
- **ImageNet normalization** — standard practice even for grayscale X-rays converted to RGB

### 3.2 Data Augmentation (Training Only)
- Random horizontal flip, rotation (±10°), brightness/contrast jitter
- Helps reduce overfitting on limited medical data

### 3.3 Class Imbalance Strategy
- **Weighted Random Sampler** — oversample minority (Normal) class
- **Weighted Cross-Entropy Loss** — penalize misclassification of minority class more

### 3.4 Model Architecture
- **ResNet50** with ImageNet pretrained weights (primary backbone)
- Freeze early layers → fine-tune `layer4` after epoch 5
- Dropout (0.5) before final classifier

### 3.5 Explainability
- **Grad-CAM:** gradient-weighted class activation mapping
- **Eigen-CAM:** first principal component of activations (often cleaner heatmaps)

In [ ]:
loaders = get_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=2 if IN_COLAB else 4)
model = build_model(backbone='resnet50', pretrained=True, dropout=0.5)
trainable, total = count_parameters(model)
print(f'ResNet50 — Trainable: {trainable:,} / Total: {total:,} parameters')

## 4. Training

Training saves the best checkpoint by **validation F1-score** (Pneumonia class).

> **Note:** Full training takes ~30–60 min on GPU. Set `RUN_TRAINING = False` to skip if a checkpoint already exists.

In [ ]:
RUN_TRAINING = True  # Set False to skip if checkpoint exists
CHECKPOINT = MODELS_DIR / 'best_resnet50.pth'

if RUN_TRAINING or not CHECKPOINT.exists():
    history = train(
        data_root=DATA_ROOT,
        backbone='resnet50',
        epochs=15,
        batch_size=BATCH_SIZE,
        lr=1e-4,
        output_dir=MODELS_DIR,
        num_workers=2 if IN_COLAB else 4,
    )
else:
    print(f'Using existing checkpoint: {CHECKPOINT}')

In [ ]:
history_path = MODELS_DIR / 'history_resnet50.json'
if history_path.exists():
    plot_training_curves(history_path, FIGURES_DIR / 'training_curves.png')
    img = plt.imread(FIGURES_DIR / 'training_curves.png')
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 5. Evaluation on Test Set

Primary metrics: **F1**, **Precision**, **Recall**, **PR-AUC**, **ROC-AUC**

In [ ]:
results = run_evaluation(
    checkpoint_path=CHECKPOINT,
    data_root=DATA_ROOT,
    output_dir=FIGURES_DIR,
    split='test',
    generate_cams=True,
    num_cam_samples=12,
    num_failure_cases=6,
)

In [ ]:
# Display key metrics
report = results['classification_report']
metrics_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': [report[c]['precision'] for c in CLASS_NAMES],
    'Recall': [report[c]['recall'] for c in CLASS_NAMES],
    'F1': [report[c]['f1-score'] for c in CLASS_NAMES],
})
metrics_df['PR-AUC'] = results['pr_auc']
metrics_df['ROC-AUC'] = results['roc_auc']
display(metrics_df)

In [ ]:
for name in ['confusion_matrix.png', 'precision_recall_curve.png', 'roc_curve.png']:
    path = FIGURES_DIR / name
    if path.exists():
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(plt.imread(path))
        ax.set_title(name.replace('_', ' ').replace('.png', '').title())
        ax.axis('off')
        plt.show()

## 6. Explainability: Grad-CAM vs Eigen-CAM

Side-by-side comparison helps radiologists understand *where* the model looks.

In [ ]:
cam_dir = FIGURES_DIR / 'cam_comparisons'
cam_images = sorted(cam_dir.glob('*.png')) if cam_dir.exists() else []

for img_path in cam_images[:6]:
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.imshow(plt.imread(img_path))
    ax.set_title(img_path.name)
    ax.axis('off')
    plt.show()

## 7. Failure Case Analysis

Understanding **false negatives** (missed pneumonia) and **false positives** (false alarms) is critical for trustworthy medical AI.

In [ ]:
failure_dir = FIGURES_DIR / 'failure_cases'
failure_images = sorted(failure_dir.glob('*.png')) if failure_dir.exists() else []

for img_path in failure_images:
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.imshow(plt.imread(img_path))
    ax.set_title(img_path.stem.replace('_', ' '))
    ax.axis('off')
    plt.show()

## 8. Key Insights & Limitations

### Insights
- Transfer learning with ResNet50 achieves strong performance on this benchmark dataset
- Eigen-CAM often produces **cleaner, less noisy** heatmaps than Grad-CAM on lung regions
- Class imbalance handling is essential — accuracy alone is misleading

### Limitations
- Dataset is from **pediatric patients** (Guangzhou) — may not generalize to adults
- Only 2 classes (Normal vs Pneumonia) — no bacterial/viral distinction
- No external validation on different hospitals/scanners
- Model confidence is not calibrated for clinical thresholds
- **Not FDA-approved or validated for clinical use**

### Future Work
1. Multi-backbone ensemble (ResNet50 + EfficientNet-B0)
2. Segmentation-guided CAM for anatomically constrained explanations
3. External dataset validation (NIH, CheXpert)